# 🤖 Model Context Protocol (MCP) Tutorial

## What You'll Learn

This notebook teaches you how to build and use **MCP (Model Context Protocol)** servers and clients. MCP is an open standard that allows AI models to securely connect to external tools and data sources.

### What is MCP?

Think of MCP like a **USB-C port for AI applications**:
- Just as USB-C lets you connect any device to your computer
- MCP lets AI models connect to any tool or data source
- It's a standardized way for AI to "plug into" external capabilities

### Architecture

```
┌─────────────────┐         ┌──────────────────┐
│   AI Client     │◄───────►│   MCP Server     │
│  (This notebook)│  MCP    │  (Tools/Data)    │
└─────────────────┘ Protocol└──────────────────┘
        │                              │
        │                              │
   Uses LLM                      Provides:
   (Kimi/Moonshot)               • Tools
                                • Resources
                                • Prompts
```

### What We're Building

1. **MCP Server** (`mcp_server.py`) - A standalone server that provides:
   - Calculator tools (add, multiply)
   - Weather lookup tool
   - Text processing utilities

2. **MCP Client** (this notebook) - Connects to the server and:
   - Discovers available tools
   - Calls tools through the MCP protocol
   - Integrates with LangChain/LangGraph


## 1️⃣ Setup: Environment and Dependencies

### Prerequisites

- Python 3.10+ installed
- `.env` file with `MOONSHOT_API_KEY` set
- The `mcp_server.py` file in your project root

### What We're Installing

| Package | Purpose |
|---------|---------|
| `fastmcp` | Build MCP servers easily |
| `mcp` | Official MCP SDK |
| `langchain-mcp-adapters` | Connect MCP to LangChain |
| `langchain-openai` | LLM integration (for Kimi API) |
| `langgraph` | Build agent workflows |

### Installation

Run the cell below to install all dependencies.

In [11]:
# Install required packages
# Run this cell first to ensure all dependencies are installed

import subprocess
import sys

print("📦 Installing MCP dependencies...")

# These versions are tested and work together
packages = [
    "fastmcp>=0.4.0",                   # MCP server creation
    "mcp>=1.0.0",                       # Official MCP SDK
    "langchain-core>=0.3.0,<0.4",       # Core LangChain
    "langchain-openai<0.3",             # OpenAI integration
    "langchain-mcp-adapters>=0.3.0",    # MCP + LangChain bridge
    "langgraph>=0.2.0",                 # Agent framework
]

for package in packages:
    print(f"   Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✅ All packages installed!")

📦 Installing MCP dependencies...
   Installing fastmcp>=0.4.0...


You should consider upgrading via the '/Users/raymaldonado/mcp_venv/bin/python -m pip install --upgrade pip' command.


   Installing mcp>=1.0.0...


You should consider upgrading via the '/Users/raymaldonado/mcp_venv/bin/python -m pip install --upgrade pip' command.


   Installing langchain-core>=0.3.0,<0.4...


...

You should consider upgrading via the '/Users/raymaldonado/mcp_venv/bin/python -m pip install --upgrade pip' command.


   Installing langchain-mcp-adapters>=0.3.0...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.2.14 requires langchain-core<0.4.0,>=0.3.27, but you have langchain-core 1.5.1 which is incompatible.
You should consider upgrading via the '/Users/raymaldonado/mcp_venv/bin/python -m pip install --upgrade pip' command.


   Installing langgraph>=0.2.0...

✅ All packages installed!


You should consider upgrading via the '/Users/raymaldonado/mcp_venv/bin/python -m pip install --upgrade pip' command.


In [12]:
# Load environment variables
# We need MOONSHOT_API_KEY for Kimi K2.5

from dotenv import load_dotenv
import os

# Load from project root (parent of notebooks folder)
env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
load_dotenv(env_path)

print("✅ Environment loaded!")

# Verify API key
api_key = os.getenv("MOONSHOT_API_KEY")
if api_key:
    print(f"✅ Moonshot API Key found: {api_key[:10]}...")
else:
    raise ValueError("❌ MOONSHOT_API_KEY not found! Add it to your .env file")

✅ Environment loaded!
✅ Moonshot API Key found: sk-TuQTfwj...


## 2️⃣ MCP Concepts

### Key Terms

| Term | Description |
|------|-------------|
| **Server** | Provides tools/resources to AI |
| **Client** | Connects to server and uses tools |
| **Tool** | Function the AI can call |
| **Transport** | How client/server communicate (we use `stdio`) |

### How It Works

```
1. Server starts and waits for connections
2. Client connects and asks: "What tools do you have?"
3. Server responds: "I have: add, multiply, get_weather..."
4. AI decides: "I need to calculate 2+2"
5. Client calls: "add(a=2, b=2)"
6. Server returns: "4"
7. AI responds to user with the result
```

### Our Server (`mcp_server.py`)

The server provides these tools:
- **Calculator**: `add`, `multiply`, `divide`, `power`
- **Weather**: `get_weather`, `get_forecast`
- **Text**: `reverse_text`, `count_words`, `to_uppercase`
- **Utils**: `generate_random_number`, `roll_dice`, `get_current_time`

## 3️⃣ MCP Server

### Server Structure

The `mcp_server.py` file defines tools using decorators:

```python
from fastmcp import FastMCP

mcp = FastMCP("TutorialServer")

@mcp.tool()
def add(a: float, b: float) -> str:
    """Add two numbers."""
    return f"{a + b}"

if __name__ == "__main__":
    mcp.run(transport='stdio')
```

### Running the Server

The server is started automatically by the MCP client - you don't need to run it manually. The client will:
1. Start the server process
2. Connect via stdio
3. Exchange messages
4. Shut down when done

## 4️⃣ MCP Client

The client connects to the server and lets us:
1. List available tools
2. Call tools directly
3. Integrate with LangChain

### Connection Code

```python
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Configure server startup
server_params = StdioServerParameters(
    command="python",
    args=["mcp_server.py"],
    env=None
)

# Connect and use tools
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        tools = await session.list_tools()
        result = await session.call_tool("add", {"a": 2, "b": 2})
```

In [ ]:
# Configure the MCP client
# This tells the client how to connect to our server

import os
import sys

print("🔧 Configuring MCP client...")

# Server is in project root (parent of notebooks folder)
server_path = os.path.join(os.path.dirname(os.getcwd()), 'mcp_server.py')

if not os.path.exists(server_path):
    raise FileNotFoundError(f"mcp_server.py not found at {server_path}")

print(f"✅ Found server at: {server_path}")

# Import MCP
from mcp import StdioServerParameters

# Configure server connection
server_params = StdioServerParameters(
    command=sys.executable,
    args=[server_path],
    env=None
)

print("✅ MCP client configured!")

🔧 Configuring MCP client...
✅ Found server at: /Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp_server.py
✅ MCP client configured!


In [ ]:
# Quick test: Verify MCP imports work

import time

print("🧪 Testing imports...")

start = time.time()
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

print(f"✅ All imports loaded in {time.time() - start:.2f}s")

🧪 Testing MCP imports...
✅ MCP imports loaded in 0.00s

🧪 Testing LangChain imports...
✅ LangChain imports successful

✅ All imports working correctly!


## 6️⃣ LangChain Integration

Now let's connect MCP tools to LangChain and use Kimi K2.5 to intelligently select and call tools.


In [15]:
# Test MCP tools directly

from mcp.client.stdio import stdio_client
from mcp import ClientSession

async def test_tools():
    """Test calculator and weather tools."""
    
    print("🧮 Testing Tools\n")
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # Test calculator
            print("1. Calculator - add(10, 32):")
            result = await session.call_tool("add", {"a": 10, "b": 32})
            print(f"   Result: {result.content[0].text}")
            
            # Test multiply
            print("\n2. Calculator - multiply(7, 6):")
            result = await session.call_tool("multiply", {"a": 7, "b": 6})
            print(f"   Result: {result.content[0].text}")
            
            # Test weather
            print("\n3. Weather - get_weather('Paris'):")
            result = await session.call_tool("get_weather", {"city": "Paris"})
            print(f"   Result: {result.content[0].text}")
            
            # Test text
            print("\n4. Text - reverse_text('Hello MCP!'):")
            result = await session.call_tool("reverse_text", {"text": "Hello MCP!"})
            print(f"   Result: {result.content[0].text}")

await test_tools()
print("\n✅ All tools working!")

🧮 Testing Tools

1. Calculator - add(10, 32):
   Result: 42.0

2. Calculator - multiply(7, 6):
   Result: 42.0

3. Weather - get_weather('Paris'):
   Result: Weather in Paris: 🌧️ Rainy, 60°F (15°C)

4. Text - reverse_text('Hello MCP!'):
   Result: !PCM olleH

✅ All tools working!


## 5️⃣ Direct Tool Testing

Let's test the MCP tools directly (without AI) to make sure everything works:


In [ ]:
# Set up LangChain with MCP tools
# This creates a ReAct agent that uses our MCP tools

from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from mcp.client.stdio import stdio_client
from mcp import ClientSession
import asyncio

# Initialize Kimi K2.5 (Moonshot API)
# Note: Kimi K2.5 only supports temperature=1
llm = ChatOpenAI(
    model="kimi-k2.5",
    base_url="https://api.moonshot.ai/v1",
    api_key=os.getenv("MOONSHOT_API_KEY"),
    temperature=1,
)

print("✅ Kimi K2.5 LLM initialized")
print("\n🔌 Connecting to MCP server...")

# Create the connection that will stay open
client_cm = stdio_client(server_params)
client = await client_cm.__aenter__()
read, write = client

# Create session that will stay open
session = ClientSession(read, write)
await session.__aenter__()
await session.initialize()

# Load all tools from the MCP server
mcp_tools = await load_mcp_tools(session)

print(f"✅ Loaded {len(mcp_tools)} tools:")
for tool in mcp_tools:
    print(f"   • {tool.name}")

# Create the ReAct agent
agent = create_react_agent(llm, mcp_tools)
print("\n✅ Agent ready!")

✅ Kimi K2.5 LLM initialized

🔌 Connecting to MCP server...
✅ Loaded 15 tools:
   • add
   • multiply
   • divide
   • power
   • calculate_temperature
   • get_weather
   • get_forecast
   • reverse_text
   • count_words
   • to_uppercase
   • to_lowercase
   • is_palindrome
   • generate_random_number
   • roll_dice
   • get_current_time

✅ Agent ready!

💡 Note: Keep this cell's context active to use the agent in subsequent cells.


/var/folders/1g/lz4110yd5_x6_lhf4pqkrphr0000gn/T/ipykernel_58407/3874304054.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools)


In [21]:
# Test the agent with a simple calculation
# The AI will decide which tool to use

async def test_agent():
    """Test the agent with a simple query."""
    
    query = "What is 25 multiplied by 17?"
    
    print(f"\n📝 Query: {query}")
    print("⏳ Thinking... (this may take 30-60 seconds)\n")
    
    try:
        # Invoke the agent
        response = await agent.ainvoke({
            "messages": [{"role": "user", "content": query}]
        })
        
        # Get the final response
        final_message = response["messages"][-1]
        print(f"🤖 Response: {final_message.content}")
        
    except Exception as e:
        import traceback
        print(f"❌ Error: {e}")
        print("\n📋 Full traceback:")
        traceback.print_exc()
        print("\n💡 Tips:")
        print("   - Check your MOONSHOT_API_KEY is valid")
        print("   - Ensure you have internet connectivity")
        print("   - The Moonshot API may be temporarily unavailable")

# Run the test
await test_agent()


📝 Query: What is 25 multiplied by 17?
⏳ Thinking... (this may take 30-60 seconds)

🤖 Response: 25 multiplied by 17 is **425**.


In [25]:
# Test with more queries

async def test_more_queries():
    """Test additional queries."""
    
    queries = [
        "What's the weather in Tokyo?",
        "What is 100 divided by 4?",
    ]
    
    for query in queries:
        print(f"\n{'='*50}")
        print(f"📝 Query: {query}")
        print("⏳ Thinking...")
        
        try:
            response = await agent.ainvoke({
                "messages": [{"role": "user", "content": query}]
            })
            final_message = response["messages"][-1]
            print(f"🤖 Response: {final_message.content}")
        except Exception as e:
            import traceback
            print(f"❌ Error: {e}")
            traceback.print_exc()

# Run the additional tests
await test_more_queries()


📝 Query: What's the weather in Tokyo?
⏳ Thinking...
🤖 Response: The weather in Tokyo is currently 🌧️ rainy with a temperature of 60°F (15°C). Don't forget to bring an umbrella if you're heading out!

📝 Query: What is 100 divided by 4?
⏳ Thinking...
🤖 Response: 100 divided by 4 equals **25**.


## 7️⃣ Debugging

### VS Code Launch Configuration

Create `.vscode/launch.json` to debug the server:

```json
{
  "version": "0.2.0",
  "configurations": [
    {
      "name": "Debug MCP Server",
      "type": "debugpy",
      "request": "launch",
      "program": "${workspaceFolder}/mcp_server.py",
      "console": "integratedTerminal"
    }
  ]
}
```

### Common Issues

| Issue | Solution |
|-------|----------|
| Import errors | Run the install cell to ensure packages are installed |
| Server not found | Check that `mcp_server.py` exists in project root |
| API errors | Verify `MOONSHOT_API_KEY` in `.env` file |
| Timeout | Kimi API may be slow; wait 30-60 seconds |

### MCP Inspector

Test your server with the MCP Inspector:

```bash
npx @modelcontextprotocol/inspector python mcp_server.py
```

## 8️⃣ Next Steps

### Extend the Server

Add more tools to `mcp_server.py`:

```python
@mcp.tool()
def count_words(text: str) -> int:
    """Count words in text."""
    return len(text.split())

@mcp.resource("config://app")
def get_config() -> dict:
    """Provide configuration data."""
    return {"version": "1.0"}
```

### Production Tips

- Use HTTP transport for remote servers
- Add authentication for security
- Implement proper error handling
- Add logging and monitoring

### Resources

- [MCP Documentation](https://modelcontextprotocol.io/)
- [FastMCP GitHub](https://github.com/jlowin/fastmcp)

## 📚 Summary

### What We Learned

1. **MCP Basics**
   - Servers provide tools, clients use them
   - Communication via stdio (local) or HTTP (remote)
   - Tools are discovered dynamically

2. **Creating Servers**
   - Use `@mcp.tool()` decorator
   - Add type hints and docstrings
   - Run with `mcp.run(transport='stdio')`

3. **Using Clients**
   - `stdio_client()` connects to local servers
   - `ClientSession` manages communication
   - Tools can be loaded into LangChain

4. **AI Integration**
   - LangChain agents can use MCP tools
   - AI automatically selects appropriate tools
   - ReAct pattern for reasoning + acting

### Files in This Tutorial

| File | Purpose |
|------|---------|
| `mcp_server.py` | MCP server with tools |
| `MCP_Tutorial.ipynb` | This notebook |
| `.env` | API keys |

### Next Steps

- Add more tools to the server
- Try building a custom agent
- Explore HTTP transport for remote deployment

## 🎯 Exercises

### Exercise 1: Add a Tool

Add to `mcp_server.py`:

```python
@mcp.tool()
def count_words(text: str) -> int:
    """Count words in text."""
    return len(text.split())
```

Then test: *"How many words are in 'The quick brown fox'?"*

### Exercise 2: Multi-Step Reasoning

Ask the agent: *"If it's 25°C in Paris, what is that in Fahrenheit?"*

The agent should use `get_weather` then `calculate_temperature`.

### Exercise 3: Custom Server

Create a new server with tools for:
- File operations (read, write)
- API calls
- Data processing

In [ ]:
# Cleanup: Close MCP connection
# Run this cell when you're done using the agent

try:
    await session.__aexit__(None, None, None)
    await client_cm.__aexit__(None, None, None)
    print("✅ MCP connection closed successfully")
except Exception as e:
    print(f"Note: {e}")